# Corrected CV: Character-Level Symbolic Recurrence Biomarker

Proper evaluation with Siamese network retrained inside each fold on train-only data.

In [ ]:
import sys
sys.path.insert(0, '..')

import pandas as pd
import numpy as np
import random
import warnings

from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import roc_auc_score
from xgboost import XGBClassifier
import tensorflow as tf

from src.pipeline import (
    preprocess_text, build_vocab, generate_all_images,
    train_siamese_and_embed
)

warnings.filterwarnings('ignore')

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
tf.random.set_seed(SEED)

## 1. Load and Preprocess

In [ ]:
# Update this path to your data file
DATA_PATH = '../data/cookie.csv'

df = pd.read_csv(DATA_PATH)

print(f"Total rows: {len(df)}")
print(f"Label distribution: {df['labels'].value_counts().to_dict()}")

df['characters'] = df['utts'].apply(preprocess_text)
char_to_index, vocab_size = build_vocab(df['characters'])
print(f"Vocabulary size: {vocab_size}")

## 2. Generate Recurrence Plot Images

In [ ]:
print("Computing recurrence plot images...")
images = generate_all_images(df, char_to_index, vocab_size, target_size=128)
labels = df['labels'].values

print(f"\nImages shape: {images.shape}")
print(f"Labels distribution: {np.bincount(labels)}")

## 3. Stratified 5-Fold CV

Siamese retrained from scratch inside each fold on train-only data.

In [ ]:
N_SPLITS = 5
skf = StratifiedKFold(n_splits=N_SPLITS, shuffle=True, random_state=SEED)

fold_aucs = []

for fold, (train_idx, test_idx) in enumerate(skf.split(images, labels)):
    print(f"\n{'='*60}")
    print(f"FOLD {fold+1}/{N_SPLITS}")
    print(f"{'='*60}")

    X_train, X_test = images[train_idx], images[test_idx]
    y_train, y_test = labels[train_idx], labels[test_idx]

    print(f"  Train: {len(train_idx)} | Test: {len(test_idx)}")
    print(f"  Train label dist: {np.bincount(y_train)}")
    print(f"  Test  label dist: {np.bincount(y_test)}")

    # train Siamese on train-only, extract embeddings
    emb_train, emb_test = train_siamese_and_embed(
        X_train, y_train, X_test, epochs=20, batch_size=16, seed=SEED
    )
    print("  Siamese trained.")

    # train XGBoost on train embeddings
    xgb = XGBClassifier(
        n_estimators=200, max_depth=5, learning_rate=0.1,
        subsample=0.8, use_label_encoder=False,
        eval_metric='logloss', random_state=SEED
    )
    xgb.fit(emb_train, y_train)

    # evaluate
    y_probs = xgb.predict_proba(emb_test)[:, 1]
    fold_auc = roc_auc_score(y_test, y_probs)
    fold_aucs.append(fold_auc)
    print(f"  Fold {fold+1} AUC: {fold_auc:.4f}")

## 4. Results with Bootstrapped 95% CI

In [ ]:
fold_aucs = np.array(fold_aucs)
mean_auc = fold_aucs.mean()
std_auc = fold_aucs.std()

n_boot = 10000
boot_means = np.array([
    np.random.choice(fold_aucs, size=N_SPLITS, replace=True).mean()
    for _ in range(n_boot)
])
ci_lo, ci_hi = np.percentile(boot_means, [2.5, 97.5])

print(f"{'='*60}")
print("RESULTS (Siamese retrained per fold, train-only pairs)")
print(f"{'='*60}")
print(f"Fold AUCs: {fold_aucs}")
print(f"Mean AUC:  {mean_auc:.4f} +/- {std_auc:.4f}")
print(f"95% CI:    [{ci_lo:.3f}, {ci_hi:.3f}]")